In [ ]:
import torch
from transformers import AutoTokenizer, BertForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("textattack/bert-base-uncased-yelp-polarity")
model = BertForSequenceClassification.from_pretrained("textattack/bert-base-uncased-yelp-polarity")

inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")

with torch.no_grad():
    logits = model(**inputs).logits

predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]

# To train a model on `num_labels` classes, you can pass `num_labels=num_labels` to `.from_pretrained(...)`
num_labels = len(model.config.id2label)
model = BertForSequenceClassification.from_pretrained("textattack/bert-base-uncased-yelp-polarity", num_labels=num_labels)

labels = torch.tensor([1])
loss = model(**inputs, labels=labels).loss
round(loss.item(), 2)

In [ ]:
from pprint import pprint

pprint(inputs)

In [ ]:
model.config

In [ ]:
# print model layers and their sizes
for name, param in model.named_parameters():
    print(name, param.size())

In [ ]:
# Load the dataset
from datasets import load_dataset

dataset = load_dataset("glue", "mrpc")


In [ ]:
dataset

In [ ]:
# Load the tokenizer and model
from transformers import AutoTokenizer, BertForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased")

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["sentence1"], examples["sentence2"], truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)


In [ ]:
tokenized_datasets

In [ ]:

# Prepare the data for PyTorch
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
# Create DataLoader
from torch.utils.data import DataLoader

train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator)
eval_dataloader = DataLoader(tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator)

In [ ]:
train_dataloader

In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Example input texts and labels
texts = ["Hello!", "How are you?"]
labels = [1, 0]

# Tokenize the input texts
encoding = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

# Add labels to the encoding
encoding["labels"] = torch.tensor(labels)

# The encoding now contains input_ids, attention_mask, and labels
pprint(encoding)

In [ ]:
import pandas as pd

# Example dataset
data = [
    {"customer_id": "C1", "baskets": [["PD_111", "PD_222"], ["PD_333", "PD_444", "PD_555"]]},
    {"customer_id": "C2", "baskets": [["PD_666", "PD_777"], ["PD_888", "PD_999", "PD_1111"]]},
]

# Convert to DataFrame
df = pd.DataFrame(data)

# Create input-output pairs
input_sequences = []
output_labels = []

for index, row in df.iterrows():
    for basket in row['baskets']:
        for i in range(1, len(basket)):
            input_sequences.append(basket[:i])
            output_labels.append(basket[i])

print("Input Sequences:", input_sequences)
print("Output Labels:", output_labels)

In [ ]:
from transformers import BertTokenizer
import torch

# Initialize a tokenizer (you may need to train a custom tokenizer for product IDs)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the input sequences
tokenized_inputs = tokenizer(input_sequences, padding=True, truncation=True, return_tensors="pt", is_split_into_words=True)

# Convert output labels to tensor
output_labels = tokenizer(output_labels, padding=True, truncation=True, return_tensors="pt", is_split_into_words=True)["input_ids"]

print("Tokenized Inputs:", tokenized_inputs)
print("Output Labels:", output_labels)